# AML_03 — Arbres, ensembles et budget de recherche

**QUID :** quelle complexité améliore le classement des fraudes pour un coût acceptable ?

Même compétition, mêmes données officielles et mêmes folds que J2. Les scores sont locaux ; aucune soumission automatique.

**Parcours renforcé.** Ouvrir cette nouvelle version au début de la séance et redémarrer le noyau. Les numéros de cellules ont changé par rapport aux premiers supports.

Chaque numéro « cellule » désigne la position physique dans ce notebook, en comptant les textes. Exécuter dans l’ordre. Installation conservée depuis le ZIP initial ; aucun git pull.

Aucune nouvelle bibliothèque.

[Guide de démarrage](https://github.com/MathieuDWeill/ai-course-student/blob/main/usecase/MyLittleMLLab/DEMARRER_J3.md) · [Cours rédigé](https://github.com/MathieuDWeill/ai-course-student/blob/main/usecase/MyLittleMLLab/lectures/AML_03_cours.md)


## Le contrat et le budget

Objectif : expliquer un gain, sa stabilité et son coût. Le budget principal est 27 fits CV, plus les exemples jouets, le contrôle et le refit d’export.

In [ ]:
from pathlib import Path
import sys, json, time, hashlib, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd()
ROOT=next((p if (p/'mllab/competition.py').exists() else p/'student') for p in [HERE,*HERE.parents] if (p/'mllab/competition.py').exists() or (p/'student/mllab/competition.py').exists())
sys.path.insert(0,str(ROOT))
from mllab.competition import subsets, make_model, xy, SPECS
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
LIMIT=int(os.environ.get('MLLAB_LIMIT','15000'))
tr,va,test,sample,protocol_record=subsets('fraud',root=ROOT,limit=LIMIT)
X,y=xy(tr,'fraud'),tr.is_fraud.astype(int)
Xv,yv=xy(va,'fraud'),va.is_fraud.astype(int)
folds=list(StratifiedKFold(3,shuffle=True,random_state=2026).split(X,y))
assert not set(tr.id)&set(va.id)
RUN=ROOT/'runs/J3';RUN.mkdir(parents=True,exist_ok=True)
ids_sha=hashlib.sha256(tr.id.to_numpy().tobytes()).hexdigest()
print('Train :',len(tr),'fraudes :',int(y.sum()),'| contrôle local :',len(va))
print('3 folds fixes ; ROC-AUC ; CPU ; même protocole que J2.')
print('Le contrôle local a déjà été consulté en J2 : il ne constitue plus un test vierge.')

plt.rcParams.update({'figure.figsize':(10,4),'font.size':14})


## Lire une règle et calculer Gini

Avant exécution, calculer les feuilles pour le seuil imposé 35 puis l’impureté pondérée. Le logiciel peut choisir un autre seuil.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
montants=np.array([10,20,30,40,50,60]).reshape(-1,1)
classes=np.array([0,0,1,0,1,1])
gauche=classes[montants[:,0]<=35];droite=classes[montants[:,0]>35]
print('Règle imposée 35 : gauche =',gauche.mean(),'droite =',droite.mean())
arbre=DecisionTreeClassifier(max_depth=1,random_state=2026).fit(montants,classes)
print('Seuil appris :',arbre.tree_.threshold[0])
plt.figure(figsize=(10,4));plot_tree(arbre,feature_names=['montant'],class_names=['normal','fraude'],filled=True,fontsize=15);plt.show()

## Profondeur et mémorisation

Prévoir le nombre de feuilles avant le calcul. Une accuracy train parfaite n’est pas une preuve de généralisation.

In [ ]:
lignes=[]
for depth in [1,2,None]:
 m=DecisionTreeClassifier(max_depth=depth,random_state=2026).fit(montants,classes)
 lignes.append({'profondeur':str(depth),'feuilles':m.get_n_leaves(),'accuracy_train_jouet':m.score(montants,classes)})
display(pd.DataFrame(lignes))
print('Trois arbres donnent .2, .5, .8 : moyenne des probabilités =',np.mean([.2,.5,.8]))

## Bootstrap et moyenne de forêt

Faire la moyenne à la main, puis lire le bootstrap. Distinguer arbres internes, observations et folds de validation.

In [ ]:
# Les lignes se répètent dans un bootstrap. Elles ne constituent pas un split de validation.
rng=np.random.default_rng(2026)
boot=rng.choice(np.arange(6),size=6,replace=True)
print('Lignes bootstrap :',boot.tolist(),'lignes distinctes :',len(set(boot)))
p=np.array([[.2,.1],[.5,.4],[.8,.7]])
print('Probabilités de trois arbres sur deux nouvelles lignes :');display(pd.DataFrame(p,columns=['ligne_A','ligne_B']))
print('Moyenne de forêt :',p.mean(axis=0))
assert np.allclose(p.mean(axis=0),[.5,.4])
print('Une forêt combine les probabilités ; les arbres ne sont pas des folds.')

## Construire trois étapes de boosting

Calculer F0=1 puis F1=[0,5,0,5,1,5,1,5]. Écrire les résidus à chaque étape. Exemple de régression séparé de la classification Kaggle.

In [ ]:
# Régression jouet pour voir les corrections. Ce n’est pas la fraude Kaggle.
from sklearn.tree import DecisionTreeRegressor
xx=np.array([1,2,3,4]).reshape(-1,1);yy=np.array([0.,0.,2.,2.])
F=np.full(4,yy.mean());eta=.5
steps=[{'etape':0,'prediction_gauche':F[0],'prediction_droite':F[-1],'mse':np.mean((yy-F)**2)}]
for m in range(1,4):
 residus=yy-F
 h=DecisionTreeRegressor(max_depth=1,random_state=2026).fit(xx,residus)
 F=F+eta*h.predict(xx)
 steps.append({'etape':m,'prediction_gauche':F[0],'prediction_droite':F[-1],'mse':np.mean((yy-F)**2)})
display(pd.DataFrame(steps))
assert np.allclose([r['mse'] for r in steps],[1,.25,.0625,.015625])
print('Régression quadratique : on ajuste les résidus y − F. En classification, le gradient de la perte remplace ce raccourci.')

## Les familles et les catégories

Hypothèse : traiter les catégories comme des catégories peut aider. Garder 80 étapes et 15 feuilles pour les deux boosting.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
num=X.select_dtypes(include='number').columns.tolist()
cat=[c for c in X if c not in num]
def native_boost(leaves=15,rate=.1):
 prep=ColumnTransformer([
  ('num',SimpleImputer(strategy='median',keep_empty_features=True),num),
  ('cat',make_pipeline(SimpleImputer(strategy='most_frequent',keep_empty_features=True),OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)),cat)])
 # Valeurs négatives inconnues traitées comme manquantes par HistGradientBoosting.
 mask=[False]*len(num)+[True]*len(cat)
 return make_pipeline(prep,HistGradientBoostingClassifier(categorical_features=mask,max_iter=80,max_leaf_nodes=leaves,learning_rate=rate,early_stopping=False,random_state=2026))
candidates={
 'logistique_C01':make_model('fraud',tr,'linear',params={'C':.1}),
 'foret_40':make_model('fraud',tr,'forest',params={'n_estimators':40,'max_depth':8}),
 'boost_ordinal':make_model('fraud',tr,'boosting',params={'early_stopping':False}),
 'boost_categoriel':native_boost()}
print('4 candidats × 3 folds = 12 fits. Catégories :',len(cat))
print('La comparaison encodage ordinal / catégories natives utilise les mêmes 80 étapes et 15 feuilles.')

## Le tournoi initial

Quatre modèles sur trois folds : 12 fits. Lire aussi les AUC train et les durées.

In [ ]:
rows=[];fold_rows=[]
for name,model in candidates.items():
 start=time.perf_counter()
 result=cross_validate(model,X,y,cv=folds,scoring='roc_auc',return_train_score=True,n_jobs=1)
 rows.append({'modele':name,'auc_cv':result['test_score'].mean(),'ecart_type':result['test_score'].std(),'auc_train':result['train_score'].mean(),'secondes':time.perf_counter()-start})
 for k,s in enumerate(result['test_score'],1):fold_rows.append({'modele':name,'fold':k,'auc':float(s)})
tournoi=pd.DataFrame(rows).sort_values('auc_cv',ascending=False)
display(tournoi.round(5));tournoi.to_csv(RUN/'tournoi.csv',index=False)
pd.DataFrame(fold_rows).to_csv(RUN/'folds.csv',index=False)

## Une expérience de représentation

Hypothèse avant calcul : comprimer quatre variables positives très étalées peut aider le modèle linéaire. Comparer remplacement et ajout de log1p, même C. Les deux essais ont un coût de six fits.

In [ ]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import make_pipeline
LOG_COLS=['amount_usd','hours_since_last_txn','distance_from_home_km','account_balance_usd']
def log_features(frame,append=False):
 out=frame.copy()
 for c in LOG_COLS:
  if (out[c].dropna()<0).any():raise ValueError('log1p demandé sur une variable négative : revoir la transformation')
  out[('log_'+c) if append else c]=np.log1p(out[c])
 return out
feature_rows=[]
for name,append in [('log_remplace',False),('log_ajoute',True)]:
 model=make_pipeline(FunctionTransformer(log_features,kw_args={'append':append},validate=False),make_model('fraud',log_features(tr,append),'linear',params={'C':.1}))
 start=time.perf_counter();res=cross_validate(model,X,y,cv=folds,scoring='roc_auc',return_train_score=True,n_jobs=1)
 feature_rows.append({'modele':name,'auc_cv':res['test_score'].mean(),'ecart_type':res['test_score'].std(),'auc_train':res['train_score'].mean(),'secondes':time.perf_counter()-start})
 candidates[name]=model
 for k,s in enumerate(res['test_score'],1):fold_rows.append({'modele':name,'fold':k,'auc':float(s)})
tournoi=pd.concat([tournoi,pd.DataFrame(feature_rows)],ignore_index=True).sort_values('auc_cv',ascending=False)
paired=pd.DataFrame(fold_rows).pivot(index='fold',columns='modele',values='auc')
paired['delta_log_remplace']=paired.log_remplace-paired.logistique_C01
paired['delta_log_ajoute']=paired.log_ajoute-paired.logistique_C01
display(tournoi.round(6));display(paired[['logistique_C01','log_remplace','log_ajoute','delta_log_remplace','delta_log_ajoute']].round(6))
tournoi.to_csv(RUN/'tournoi.csv',index=False);paired.to_csv(RUN/'comparaison_pairee.csv')
print('Deux représentations × trois folds = six fits supplémentaires. Même C=0,1 et mêmes observations.')

## Une recherche bornée

Choisir trois configurations motivées. Ne pas agrandir la grille après lecture des scores pour chercher une victoire.

In [ ]:
configs=[{'leaves':7,'rate':.1},{'leaves':15,'rate':.1},{'leaves':15,'rate':.03}]
search_rows=[]
for cfg in configs:
 start=time.perf_counter();res=cross_validate(native_boost(**cfg),X,y,cv=folds,scoring='roc_auc',n_jobs=1)
 search_rows.append({**cfg,'auc_cv':res['test_score'].mean(),'ecart_type':res['test_score'].std(),'secondes':time.perf_counter()-start})
recherche=pd.DataFrame(search_rows);display(recherche.round(5))
recherche.to_csv(RUN/'recherche.csv',index=False)
print('Budget réalisé : 12 + 6 + 9 = 27 fits CV. Les neuf fits incluent une répétition de la configuration de référence.')

## La décision arrêtée avant le contrôle

Les six candidats et les trois réglages natifs ont désormais été comparés en CV. Le choix ne dépend pas du contrôle local.

In [ ]:
best_cfg=recherche.sort_values('auc_cv',ascending=False).iloc[0]
best_base=tournoi.iloc[0]
if best_cfg.auc_cv>best_base.auc_cv:
 selected_name='boost_categoriel_regle';selected_model=native_boost(int(best_cfg.leaves),float(best_cfg.rate))
else:
 selected_name=str(best_base.modele);selected_model=clone(candidates[selected_name])
print('Candidat retenu par la CV :',selected_name)
selected_model.fit(X,y)
controle=roc_auc_score(yv,selected_model.predict_proba(Xv)[:,1])
print('AUC du contrôle local déjà utilisé en J2 :',round(controle,6))
print('Ce résultat est descriptif ; pas une estimation indépendante finale.')

## L’importance par permutation

Une association prédictive n’est pas une relation causale. Lire l’effet des variables corrélées et la limite des deux répétitions.

In [ ]:
from sklearn.inspection import permutation_importance
imp=permutation_importance(selected_model,Xv,yv,scoring='roc_auc',n_repeats=2,random_state=2026,n_jobs=1)
importance=pd.DataFrame({'variable':X.columns,'baisse_auc':imp.importances_mean,'dispersion':imp.importances_std}).sort_values('baisse_auc',ascending=False)
display(importance.head(8).round(5));importance.to_csv(RUN/'importance.csv',index=False)
print('Variables corrélées : une permutation isolée peut sous-estimer leur utilité commune.')

## L’audit des sous-groupes

Examiner les types de paiements, les effectifs et les erreurs au seuil 0,5. Ne pas sélectionner un seuil à partir de cet audit.

In [ ]:
from sklearn.metrics import confusion_matrix
pred_control=selected_model.predict_proba(Xv)[:,1]
audit=va[['id','channel','is_foreign_transaction']].copy()
audit['cible']=yv.to_numpy();audit['score']=pred_control
slice_rows=[]
for col in ['channel','is_foreign_transaction']:
 for value,part in audit.groupby(col,dropna=False):
  tn,fp,fn,tp=confusion_matrix(part.cible,part.score>=.5,labels=[0,1]).ravel()
  slice_rows.append({'variable':col,'groupe':str(value),'lignes':len(part),'fraudes':int(part.cible.sum()),'auc':roc_auc_score(part.cible,part.score) if part.cible.nunique()==2 else np.nan,'rappel_05':tp/(tp+fn) if tp+fn else np.nan,'fausses_alertes':int(fp)})
slice_table=pd.DataFrame(slice_rows);display(slice_table.round(4));slice_table.to_csv(RUN/'audit_sous_groupes.csv',index=False)
print('Petits groupes : résultats fragiles. Cet audit suggère des hypothèses, sans régler le modèle sur ce contrôle déjà connu.')

## Le candidat et le CSV Kaggle

Refit après le choix, puis prédiction sur tout le test officiel. Aucun envoi automatique.

In [ ]:
from sklearn.model_selection import train_test_split
from mllab.competition import read_competition
all_train,test,sample=read_competition('fraud',root=ROOT)
fit_train,_=train_test_split(all_train,train_size=LIMIT,stratify=all_train.is_fraud,random_state=2026)
final_model=clone(selected_model).fit(xy(fit_train,'fraud'),fit_train.is_fraud)
pred=final_model.predict_proba(xy(test,'fraud'))[:,1]
submission=sample.copy();submission['is_fraud']=pred
assert list(submission)==['id','is_fraud'] and np.array_equal(submission.id,test.id)
assert np.isfinite(pred).all() and ((pred>=0)&(pred<=1)).all()
submission.to_csv(RUN/'submission_J3.csv',index=False)
manifest={'modele':selected_name,'parametres':str(selected_model.get_params()),'train_ids_sha256':ids_sha,'donnees_sha256':protocol_record['data_sha256'],'fits_cv':27,'controle_auc':controle,'lignes_submission':len(submission),'submission_sha256':hashlib.sha256((RUN/'submission_J3.csv').read_bytes()).hexdigest(),'score_public':None}
(RUN/'manifest.json').write_text(json.dumps(manifest,indent=2))
display(submission.head());print(len(submission),'lignes exportées ; aucun envoi à Kaggle.')

## Le journal et la décision

Remplacer les champs à compléter par une hypothèse, deux chiffres, une limite et une prochaine expérience à budget fixé.

In [ ]:
journal={'hypothese':'À compléter avant une nouvelle expérience','preuve':'À compléter avec deux scores et un temps','decision':'À compléter : conserver ou abandonner, et pourquoi','prochaine_experience':'Un seul changement et un budget annoncé','score_public_reel':None}
(RUN/'journal.json').write_text(json.dumps(journal,indent=2,ensure_ascii=False))
print('Fichiers :',', '.join(sorted(p.name for p in RUN.iterdir() if p.is_file())))

## La vérification finale

Un fichier valide ne prouve pas un bon modèle. Vérifier le format puis présenter une conclusion de deux minutes.

In [ ]:
checks={'meme_train':len(X)==len(y),'auc_complete':tournoi.auc_cv.notna().all(),'27_fits_cv':27==12+6+9,'format_csv':list(submission)==['id','is_fraud'],'ordre_ids':np.array_equal(submission.id,test.id),'scores_finis':np.isfinite(submission.is_fraud).all()}
assert all(checks.values());display(pd.Series(checks,name='controle'))
print('À expliquer : 1) forêt ou boosting ? 2) quel changement a réellement aidé ? 3) le gain est-il régulier sur les folds ?')

## Remise

Conserver les tableaux, le manifeste et le CSV. Compléter le journal avec une hypothèse, une preuve chiffrée, une décision et une prochaine expérience bornée. Préparer une explication orale de deux minutes. Le contrôle local est déjà connu depuis J2. Le test officiel n’a pas de cible accessible et aucun score public n’est inventé.